# 04 — Parallelism, Kernels, and Synchronization

**Master syllabus coverage:** V2 1.4

## Why this matters

Fast tensor libraries divide operations among parallel workers. Correct indexing, ownership, synchronization, and launch granularity are prerequisites for understanding GPU and future WGSL engine work.

## Learning objectives

- Map one output operation to conceptual kernel invocations and workgroups.
- Identify independent work, dependencies, reductions, barriers, and races.
- Explain launch overhead and operation-fusion opportunities.
- Relate workgroup size, divergence, and utilization without assuming one universal optimum.

Work through the notebook in order. Predict shapes and results before running each code cell. An assertion failure is part of the lesson: read it before changing code.


## 1. A kernel applies one program across many indices

Host code launches a kernel; many invocations compute different output elements. CPUs
expose threads and SIMD vectors; GPUs organize SIMT lanes into warps/wavefronts and
workgroups. Physical execution details vary, but every parallel algorithm must define
work partitioning, memory ownership, and dependencies.


In [ ]:
import time
import numpy as np
import torch

left = np.arange(17, dtype=np.float32)
right = np.ones_like(left)

def vector_add_invocation(index: int, a: np.ndarray, b: np.ndarray, out: np.ndarray) -> None:
    if index < out.size:  # boundary guard for rounded-up workgroups
        out[index] = a[index] + b[index]

output = np.empty_like(left)
workgroup_size = 8
launched_invocations = ((left.size + workgroup_size - 1) // workgroup_size) * workgroup_size
for index in range(launched_invocations):
    vector_add_invocation(index, left, right, output)
np.testing.assert_array_equal(output, left + right)
print("elements:", left.size, "launched invocations:", launched_invocations)


## 2. Parallel writes need ownership or coordination

Independent elementwise outputs require no barrier because each worker owns one index.
Reductions make many workers contribute to shared state and require staged partial sums,
atomics, locks, or another coordination design. A race condition means the result depends
on unpredictable execution interleaving.


In [ ]:
values = np.arange(1, 17, dtype=np.float32)
groups = values.reshape(4, 4)
partial_sums = groups.sum(axis=1)  # each conceptual workgroup owns one result
total = partial_sums.sum()         # second synchronized stage
assert total == values.sum()
print("stage-one partial sums:", partial_sums, "stage-two total:", total)

# Lost-update illustration: two workers both read 0, then both write 1.
shared = 0
worker_a_read = shared
worker_b_read = shared
shared = worker_a_read + 1
shared = worker_b_read + 1
print("racy result:", shared, "expected serialized result: 2")
assert shared == 1


## 3. Launches and synchronization have fixed overhead

Many tiny operations can spend more time in Python/framework dispatch and kernel launch
than arithmetic. Fusion combines compatible operations, reduces intermediate memory
traffic, and reduces launches. Fusion can increase compilation complexity or register use,
so measure the real shapes.


In [ ]:
x = torch.randn(200_000)
repeats = 100

start = time.perf_counter()
for _ in range(repeats):
    a = x + 1
    b = a * 2
    separate = torch.relu(b)
separate_time = time.perf_counter() - start

start = time.perf_counter()
for _ in range(repeats):
    # One expression is not guaranteed one physical kernel, but exposes fusion opportunity.
    fused_candidate = torch.relu((x + 1) * 2)
candidate_time = time.perf_counter() - start
torch.testing.assert_close(separate, fused_candidate)
print(f"separate expressions={separate_time:.4f}s, fusion candidate={candidate_time:.4f}s")


## 4. Dependencies bound parallelism

A prefix sum has a sequential-looking recurrence; specialized scan algorithms reorganize
it into parallel stages. Autoregressive decoding likewise cannot generate token `t+1`
before selecting token `t`, although work *inside* one token step is highly parallel.
Training knows all target tokens and computes sequence positions in parallel under a mask.


In [ ]:
sequence = torch.arange(1, 9)
serial = []
running = 0
for value in sequence:
    running += int(value)
    serial.append(running)
parallel_library = torch.cumsum(sequence, dim=0)
assert serial == parallel_library.tolist()
print("prefix sum:", serial)


## 5. Workgroup size is a hardware-informed tuning parameter

Too little work wastes lanes and launch capacity; too much per group can exhaust registers
or local/shared memory and reduce occupancy. Divergent branches can make lanes execute
both paths while subsets remain idle. Portable code starts correct, then measures candidate
configurations on target hardware.


In [ ]:
elements = 1_000
for group_size in (32, 64, 128, 256):
    groups = (elements + group_size - 1) // group_size
    launched = groups * group_size
    utilization = elements / launched
    print(f"group={group_size:3}: groups={groups:2}, boundary utilization={utilization:.3f}")


## Failure modes to recognize

- **Out-of-bounds invocation:** rounded-up launch sizes access invalid storage.
- **Race condition:** multiple workers update shared state without coordination.
- **Missing synchronization:** a later stage reads incomplete partial results.
- **Tiny-kernel storm:** fixed dispatch and memory costs dominate arithmetic.

These are diagnostic patterns, not trivia. When a future model fails, reduce the problem to the smallest example in this notebook and test the relevant invariant.


## Deliberate practice

1. Design a two-stage parallel maximum reduction including ownership and synchronization points.
2. Count launches and temporary arrays in a small unfused activation pipeline.
3. Explain which parts of autoregressive generation are sequential and which remain parallel.

Do not merely rerun the provided cells. Make a prediction, change one variable, and write down why the result did or did not match your prediction.

## Exit ticket

**You are ready to continue when:** you can partition an operation among workers, state who writes every location, and identify necessary synchronization and launch costs.

**Next:** 05 — Matrix multiplication as a systems problem.
